# Wirewalker ADCP L2 explorer

Interactive exploration of a gridded `(depth, cast)` velocity product built by
`process_ww_sig1000.py --product velocity`.

### Two things this notebook does deliberately

**Upcasts only.** Down casts carry a bias that *grows with depth*. On NOPP_d2 the two
directions differ by **0.116 m/s at 480 m** — about twice the median signal, and
opposite in sign — and averaging them together cancels the real deep flow into a
muddled near-zero field. Mean vertical velocity identifies upcasts as the correct
population: `velU` is +0.0005 m/s at 480 m on upcasts (the ocean's true value is ~0)
against −0.0112 m/s on downcasts. The load cell below subsets to `cast_direction == 1`.

**Scatter, never a section.** Every marker is one real `(cast, depth-bin)` measurement.
Nothing is gridded, interpolated, smoothed or filled between casts, so duty-cycle gaps
stay genuinely empty and irregular cast spacing shows as it is. When a selection holds
more points than the cap, they are *decimated* — every Nth point — never averaged, and
both counts are printed in the title so you always know what you are looking at.

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import xarray as xr
import ipywidgets as W

# --- point this at the product you want to explore -------------------------------
L2_PATH = Path(
    "~/NOPP/pyNOPP/nopp1-california/S100430A038_NOPP_d2/processed/"
    "nopp_d2_sig1000_L2up_v2s2_grid1m.nc"
).expanduser()
# ---------------------------------------------------------------------------------

TEMPLATE = "plotly_white"
# categorical slots of the project's validated palette, used in fixed order
C1, C2, C3 = "#2a78d6", "#eb6834", "#1baf7a"

print(L2_PATH)
print("exists:", L2_PATH.exists())

In [ ]:
ds_all = xr.open_dataset(L2_PATH)
ds = ds_all.isel(cast=(ds_all["cast_direction"].values == 1))   # upcasts only

t = pd.DatetimeIndex(ds["time"].values)
z = ds["depth"].values
comp = ds["profile_complete"].values

n_drop = ds_all.sizes["cast"] - ds.sizes["cast"]
print(f"{ds_all.sizes['cast']:,} casts in file  ->  {ds.sizes['cast']:,} upcasts kept "
      f"({n_drop:,} downcasts dropped)")
print(f"depth   {z.min():.1f} - {z.max():.1f} m  in {z.size} bins of {np.diff(z)[0]:.2f} m")
print(f"time    {t[0]:%Y-%m-%d %H:%M}  ->  {t[-1]:%Y-%m-%d %H:%M}   ({(t[-1] - t[0]).days} days)")
print(f"casts   {int((comp == 1).sum()):,} complete, {int((comp == 0).sum()):,} truncated")
print(f"vars    {list(ds.data_vars)}")
ds

## Selecting points

`points()` flattens the `(depth, cast)` array to one row per real measurement, keeping
only finite values. Every downstream plot is built from its output, so no plot can
accidentally introduce a filled or interpolated value.

In [ ]:
def points(var, t0=None, t1=None, zmin=None, zmax=None, complete_only=False):
    '''Flatten (depth, cast) to one row per finite measurement.

    Returns (times, depths, values, complete_flag) as 1-D arrays of equal length.
    Subsetting only ever drops points - it never creates or averages them.
    '''
    sel = ds
    if complete_only:
        sel = sel.isel(cast=(sel["profile_complete"].values == 1))

    tt = pd.DatetimeIndex(sel["time"].values)
    keep = np.ones(tt.size, bool)
    if t0 is not None:
        keep &= tt >= pd.Timestamp(t0)
    if t1 is not None:
        keep &= tt <= pd.Timestamp(t1)
    sel = sel.isel(cast=keep)

    zz = sel["depth"].values
    keep_z = np.ones(zz.size, bool)
    if zmin is not None:
        keep_z &= zz >= zmin
    if zmax is not None:
        keep_z &= zz <= zmax
    sel = sel.isel(depth=keep_z)

    a = sel[var].values                                   # (nz, ncast)
    nz, nc = a.shape
    T = np.tile(sel["time"].values, (nz, 1))
    Z = np.repeat(sel["depth"].values[:, None], nc, axis=1)
    C = np.tile(sel["profile_complete"].values, (nz, 1))

    m = np.isfinite(a)
    return T[m], Z[m], a[m], C[m]


DIVERGING = {"velE", "velN", "velU"}       # signed fields get a two-hue scale


def scatter(var="velE", t0=None, t1=None, zmin=None, zmax=None, clip=None,
            complete_only=False, max_points=150_000, size=3):
    '''Depth-time scatter of individual measurements. No interpolation anywhere.'''
    T, Z, V, C = points(var, t0, t1, zmin, zmax, complete_only)
    n_avail = V.size
    if n_avail == 0:
        print("no measurements in this selection")
        return None
    step = max(1, int(np.ceil(n_avail / max_points)))
    T, Z, V, C = T[::step], Z[::step], V[::step], C[::step]

    if var in DIVERGING:
        lim = clip if clip else float(np.nanpercentile(np.abs(V), 99))
        ckw = dict(cmin=-lim, cmax=lim, colorscale="RdBu", reversescale=True)
    else:
        ckw = dict(colorscale="Viridis")

    units = ds[var].attrs.get("units", "")
    fig = go.Figure(go.Scattergl(
        x=T, y=Z, mode="markers",
        marker=dict(color=V, size=size, opacity=0.85, line=dict(width=0),
                    colorbar=dict(title=f"{var}<br>{units}", thickness=14), **ckw),
        customdata=C,
        hovertemplate=("%{x|%Y-%m-%d %H:%M}<br>%{y:.1f} m<br>"
                       + f"{var} = " + "%{marker.color:.4f}<br>"
                       "complete = %{customdata}<extra></extra>"),
    ))
    fig.update_yaxes(autorange="reversed", title="depth (m)")
    fig.update_xaxes(title="time (UTC)")
    shown = "all" if step == 1 else f"every {step}"
    fig.update_layout(
        template=TEMPLATE, height=560, margin=dict(l=60, r=10, t=60, b=50),
        title=(f"{var} &mdash; upcasts only &nbsp;|&nbsp; {n_avail:,} measurements, "
               f"{V.size:,} plotted ({shown}) &nbsp;|&nbsp; no interpolation"))
    return fig


fig = scatter()
fig.show()

## Interactive controls

Set the selection, then press **Run Interact** to redraw. Nothing updates until you
press it — with up to a million points a redraw is not free.

- **max pts** caps how many markers are drawn. Raise it to see everything; lower it for
  a responsive first look. The title always reports the decimation.
- **complete casts only** drops profiles clipped by a duty-cycle burst boundary. They
  are valid data over the depths they cover (they differ from complete casts by
  ≤0.005 m/s), so this is usually worth leaving off.
- Plotly's own zoom/pan/box-select work on the result; double-click resets.

In [ ]:
days = pd.date_range(t[0].floor("D"), t[-1].ceil("D"), freq="1D")
date_opts = [(d.strftime("%Y-%m-%d"), d) for d in days]
wide = W.Layout(width="92%")

controls = dict(
    var=W.Dropdown(options=list(ds.data_vars), value="velE", description="variable"),
    rng=W.SelectionRangeSlider(options=date_opts, index=(0, len(date_opts) - 1),
                               description="dates", continuous_update=False, layout=wide),
    zr=W.IntRangeSlider(value=(0, int(z.max()) + 1), min=0, max=int(z.max()) + 1, step=5,
                        description="depth m", continuous_update=False, layout=wide),
    clip=W.FloatSlider(value=0.25, min=0.02, max=1.0, step=0.01, description="colour +/-",
                       continuous_update=False, readout_format=".2f"),
    complete_only=W.Checkbox(value=False, description="complete casts only"),
    max_points=W.Dropdown(options=[25_000, 50_000, 150_000, 400_000, 1_000_000],
                          value=150_000, description="max pts"),
    size=W.FloatSlider(value=3, min=1, max=8, step=0.5, description="marker"),
)


def _draw(var, rng, zr, clip, complete_only, max_points, size):
    f = scatter(var=var, t0=rng[0], t1=rng[1], zmin=zr[0], zmax=zr[1], clip=clip,
                complete_only=complete_only, max_points=max_points, size=size)
    if f is not None:
        f.show()


W.interact_manual(_draw, **controls);

## One cast at a time

Individual upcast profiles, markers at each depth bin. Truncated casts simply stop
where the burst ended — the gap is real, not missing data.

In [ ]:
def profile(i, variables=("velE", "velN", "velU")):
    fig = go.Figure()
    for v, colour in zip(variables, (C1, C2, C3)):
        fig.add_trace(go.Scatter(x=ds[v].values[:, i], y=z, mode="markers",
                                 name=v, marker=dict(size=4, color=colour)))
    ts = pd.Timestamp(ds["time"].values[i])
    flag = "complete" if ds["profile_complete"].values[i] else "TRUNCATED"
    span = f"{ds['pressure_min'].values[i]:.0f}-{ds['pressure_max'].values[i]:.0f} dbar"
    fig.update_yaxes(autorange="reversed", title="depth (m)")
    fig.update_xaxes(title="velocity (m s-1)", zeroline=True, zerolinewidth=1)
    fig.update_layout(template=TEMPLATE, height=620, hovermode="y unified",
                      margin=dict(l=60, r=10, t=60, b=50),
                      title=f"upcast {i} &mdash; {ts:%Y-%m-%d %H:%M} &nbsp;|&nbsp; {flag} &nbsp;|&nbsp; {span}")
    return fig


W.interact(lambda i: profile(i).show(),
           i=W.IntSlider(0, 0, ds.sizes["cast"] - 1, description="cast",
                         continuous_update=False, layout=wide));

## Depth-mean time series

A quick orientation view: depth-averaged components and speed, binned in time. This one
*does* average (over depth, and into time bins) — it is a summary, not the measurement
view above.

In [ ]:
FREQ = "1D"          # any pandas offset: "6h", "1D", "7D", ...

with warnings.catch_warnings():
    warnings.filterwarnings("ignore", "Mean of empty slice")
    speed = np.sqrt(ds["velE"].values ** 2 + ds["velN"].values ** 2)
    frame = pd.DataFrame(
        {"velE": np.nanmean(ds["velE"].values, axis=0),
         "velN": np.nanmean(ds["velN"].values, axis=0),
         "speed": np.nanmean(speed, axis=0)},
        index=t).resample(FREQ).mean()

fig = go.Figure()
for name, colour in (("velE", C1), ("velN", C2), ("speed", C3)):
    fig.add_trace(go.Scatter(x=frame.index, y=frame[name], name=name,
                             line=dict(color=colour, width=2)))
fig.add_hline(y=0, line_width=1, line_color="#9aa3ab")
fig.update_layout(template=TEMPLATE, height=400, hovermode="x unified",
                  margin=dict(l=60, r=10, t=60, b=50),
                  yaxis_title="m s-1", xaxis_title="time (UTC)",
                  title=f"Depth-mean velocity, {FREQ} means (upcasts, {ds.sizes['cast']:,} casts)")
fig.show()

## Check the choice yourself

The upcast-only decision is load-bearing, so here is the comparison it rests on, run
against this file rather than quoted. `velU` is the independent test: the ocean's mean
vertical velocity is ~0, and only one of these populations reproduces that.

In [ ]:
up = ds_all["cast_direction"].values == 1
rows = []
with warnings.catch_warnings():
    warnings.filterwarnings("ignore", "Mean of empty slice")
    for zt in (25, 50, 100, 200, 300, 400, 480):
        i = int(np.argmin(np.abs(ds_all["depth"].values - zt)))
        row = {"depth_m": round(float(ds_all["depth"].values[i]))}
        for v in ("velE", "velN", "velU"):
            a = ds_all[v].values[i]
            row[f"{v}_up"] = np.nanmean(a[up])
            row[f"{v}_dn"] = np.nanmean(a[~up])
        row["velN_diff"] = row["velN_up"] - row["velN_dn"]
        rows.append(row)

cmp = pd.DataFrame(rows).set_index("depth_m")
print(cmp.round(4).to_string())
print("\nDeployment mean velU   up: {:+.4f}   down: {:+.4f}   (ocean truth ~ 0)".format(
    np.nanmean(cmp["velU_up"]), np.nanmean(cmp["velU_dn"])))

# Turbulence (HR beam-5 spectral ε)

Self-contained section — run from here without the cells above. The open question
(2026-08-21): **is ε below ~100 m meaningful?** Evidence for a noise floor: beam-5 corr
falls to ~48% at depth (below `corr_min = 50`), data return 38% in the deepest bins, the
spectral noise floor `N` rises ~0.7 dex with depth while SNR falls 28→19, only ~12
wavenumbers resolved at this 0.10 m cell size, and the full-record profile is
suspiciously flat below 100 m. **Counter-evidence**: May's 300–500 m median was −8.26,
well below that apparent floor — the instrument *can* resolve lower values at depth.

In [ ]:
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

TB = xr.open_dataset("/Users/drew/NOPP/pyNOPP/nopp1-california/S100430A038_NOPP_d2/"
                     "processed/nopp_d2_sig1000_turb_dep3m.nc")
tt = pd.to_datetime(TB.time.values)
tz = TB.depth.values
leps = np.log10(TB.epsilon.values)
print(f"{TB.sizes['cast']} casts x {TB.sizes['depth']} depths, "
      f"{tt[0].date()} .. {tt[-1].date()};  finite eps: {np.isfinite(leps).mean():.0%}")

fig, axes = plt.subplots(2, 1, figsize=(13, 7), sharex=True, constrained_layout=True,
                         height_ratios=[2.2, 1])
pc = axes[0].pcolormesh(tt, tz, leps, vmin=-9.5, vmax=-5.5, cmap="magma", shading="nearest")
axes[0].set_ylim(500, 0); axes[0].set_ylabel("depth (m)")
fig.colorbar(pc, ax=axes[0], pad=0.01, label="log$_{10}$ ε (W/kg)")
for z0, z1, col in ((10, 100, "tab:orange"), (300, 500, "tab:blue")):
    m = (tz >= z0) & (tz < z1)
    s = pd.Series(np.nanmedian(leps[m], axis=0), index=tt).resample("2D").median()
    axes[1].plot(s.index, s.values, color=col, lw=1.4, label=f"{z0}-{z1} m")
axes[1].set_ylabel("log$_{10}$ ε"); axes[1].legend()
axes[1].xaxis.set_major_formatter(mdates.DateFormatter("%b"))
axes[0].set_title("ε section and band medians — note May deep minimum vs the June regime change", loc="left");

## Floor anatomy: what sets the deep values?
If the flat deep profile is a noise floor, ε at depth should be **pinned wherever SNR is
low** and track the spectral noise `N`; if it is ocean, ε should vary independently of
data-quality metrics. Left: full-record median profiles. Right: log ε vs SNR by depth
band — a floor shows up as the deep cloud collapsing onto a line at low SNR.

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(14, 6), sharey=True, constrained_layout=True)
profs = [("epsilon", np.nanmedian(leps, axis=1), "log$_{10}$ ε"),
         ("SNR", np.nanmedian(TB.SNR.values, axis=1), "SNR"),
         ("corr", np.nanmedian(TB.corr.values, axis=1), "beam-5 corr (%)"),
         ("N", np.nanmedian(np.log10(TB.N.values), axis=1), "log$_{10}$ N"),
         ("num_spectra", np.nanmedian(TB.num_spectra.values, axis=1), "spectra/bin")]
for ax, (name, p, lab) in zip(axes, profs):
    ax.plot(p, tz, lw=1.5)
    ax.set_xlabel(lab); ax.grid(alpha=0.3)
    if name == "corr":
        ax.axvline(50, color="r", lw=0.8, ls="--")
axes[0].set_ylim(500, 0); axes[0].set_ylabel("depth (m)")
fig.suptitle("full-record median profiles — the deep flattening of ε against its quality metrics");

fig2, axes2 = plt.subplots(1, 3, figsize=(13, 4.5), sharey=True, constrained_layout=True)
for ax, (z0, z1) in zip(axes2, ((10, 100), (100, 300), (300, 500))):
    m = (tz >= z0) & (tz < z1)
    x = TB.SNR.values[m].ravel(); y = leps[m].ravel()
    ok = np.isfinite(x) & np.isfinite(y)
    ax.hexbin(x[ok], y[ok], gridsize=45, bins="log", cmap="viridis", extent=(0, 60, -10, -5))
    ax.set_title(f"{z0}-{z1} m"); ax.set_xlabel("SNR")
axes2[0].set_ylabel("log$_{10}$ ε")
fig2.suptitle("ε vs SNR by depth band — is deep ε pinned at low SNR?");

In [ ]:
# distributions: a real lognormal-ish ocean signal vs a hard noise-floor edge,
# split May (deep minimum) vs July (post regime change) to test whether the
# apparent floor moves with the season or stays fixed (a fixed floor = instrument).
fig, axes = plt.subplots(1, 3, figsize=(13, 4), sharey=True, constrained_layout=True)
mo = tt.month
for ax, (z0, z1) in zip(axes, ((10, 100), (100, 300), (300, 500))):
    mz = (tz >= z0) & (tz < z1)
    for sel, lab, col in ((mo == 5, "May", "tab:blue"), (mo == 7, "Jul", "tab:red")):
        x = leps[np.ix_(mz, sel)].ravel()
        x = x[np.isfinite(x)]
        ax.hist(x, bins=np.linspace(-10.5, -5, 80), density=True, histtype="step",
                lw=1.6, color=col, label=f"{lab} (med {np.median(x):.2f})")
    ax.set_title(f"{z0}-{z1} m"); ax.set_xlabel("log$_{10}$ ε"); ax.legend(fontsize=8)
axes[0].set_ylabel("pdf");

## Two-week ε browser
Same controls as the velocity browser: `start` picks the window, `vmax`/`vmin` the color
range, the vertical slider the depth range. Top: ε; bottom: SNR (its quality shadow).

In [ ]:
import ipywidgets as W
from IPython.display import display

tb_starts = pd.date_range(tt[0].normalize(), tt[-1] - pd.Timedelta(days=14), freq="D")
tb_date = W.SelectionSlider(options=[(d.strftime("%Y-%m-%d"), d) for d in tb_starts],
                            description="start", continuous_update=False,
                            layout=W.Layout(width="95%"))
tb_depth = W.IntRangeSlider(value=[0, 500], min=0, max=500, step=10,
                            description="depth (m)", orientation="vertical",
                            continuous_update=False, layout=W.Layout(height="340px"))

def tb_window(start, zrange):
    sel = (tt >= start) & (tt < start + pd.Timedelta(days=14))
    if sel.sum() < 5:
        print("no casts in this window"); return
    tw = tt[sel]
    fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True, sharey=True,
                             constrained_layout=True)
    pc0 = axes[0].pcolormesh(tw, tz, leps[:, sel], vmin=-9.5, vmax=-5.5,
                             cmap="magma", shading="nearest")
    fig.colorbar(pc0, ax=axes[0], pad=0.01, label="log$_{10}$ ε")
    pc1 = axes[1].pcolormesh(tw, tz, TB.SNR.values[:, sel], vmin=0, vmax=40,
                             cmap="viridis", shading="nearest")
    fig.colorbar(pc1, ax=axes[1], pad=0.01, label="SNR")
    axes[0].set_ylim(zrange[1], zrange[0])
    for ax, ttl in zip(axes, ("ε", "SNR")):
        ax.set_ylabel("depth (m)"); ax.set_title(ttl, loc="left", fontsize=10)
    axes[1].xaxis.set_major_formatter(mdates.DateFormatter("%m-%d"))
    fig.suptitle(f"{start:%Y-%m-%d} + 14 d  ({sel.sum()} casts)")
    plt.show()

tb_out = W.interactive_output(tb_window, {"start": tb_date, "zrange": tb_depth})
display(W.HBox([W.VBox([tb_date, tb_out], layout=W.Layout(width="90%")), tb_depth]))